### Extract state vector
Import libraries

In [1]:
import sys

libs = {
  "./../modules",
  "/u/zanardi3/Codes/ML/PyCOMET/pycomet"
}
for lib in libs:
  sys.path.append(lib)

In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
import reading as read_mod

from converter import Converter

Define inputs

In [4]:
identifier = "vel1_i10"
# Path to database
path_to_dtb = "./../../database/"
repeats = 3
# Path to saved solution
path_to_sol = f"./../output/data/{identifier}.p"
# Path to saving folder
path_to_save = f"./../output/sampling/{identifier}/init_sols/"

In [5]:
os.makedirs(path_to_save, exist_ok=True)

Read saved solution

In [6]:
dset = read_mod.load_dataset(path_to_sol)

> Extract post-shock states

In [7]:
i_ps = dset["p"] > dset["p"][0,0]
for (k, v) in dset.items():
  if (k not in ("x_grid", "t_grid")):
    dset[k] = v[i_ps]

> Get primitive variables

In [8]:
rho, Th, Te = [np.expand_dims(dset[k], axis=-1) for k in ("rho", "Th", "Te")]
rho = dset["w"] * rho
prim = np.concatenate([rho, Th, Te], axis=-1)
prim = np.repeat(prim, repeats, axis=0)

Save data

In [9]:
conv = Converter(path_to_dtb)

In [10]:
df_smp = pd.DataFrame(prim, columns=conv.prim_lbl)
df_smp.to_csv(path_to_save + "/train.csv", float_format="%.8e", index=False)